In [ ]:
# 5. Verificar outputs salvos
from src.io import paths

DATA_PATHS = paths.get_data_paths()
PROC_PATH = str(DATA_PATHS["data_processed"])

news_clean_file = os.path.join(PROC_PATH, "news_clean.parquet")
bow_daily_file = os.path.join(PROC_PATH, "bow_daily.parquet")

print("\n" + "="*60)
print("📦 OUTPUTS GERADOS")
print("="*60)

if os.path.exists(news_clean_file):
    size_mb = os.path.getsize(news_clean_file) / (1024*1024)
    print(f"\n✅ news_clean.parquet")
    print(f"   Path: {news_clean_file}")
    print(f"   Size: {size_mb:.2f} MB")
    print(f"   Rows: {len(df_clean):,}")

if os.path.exists(bow_daily_file):
    df_bow = pd.read_parquet(bow_daily_file)
    size_mb = os.path.getsize(bow_daily_file) / (1024*1024)
    print(f"\n✅ bow_daily.parquet")
    print(f"   Path: {bow_daily_file}")
    print(f"   Size: {size_mb:.2f} MB")
    print(f"   Days: {len(df_bow):,}")
    print(f"   Avg news/day: {df_bow['news_count'].mean():.1f}")

print("\n⏭️ Próximo passo: Executar Notebook 15 (Features TF-IDF Diário)")

In [ ]:
# 4. Análise de tokens e vocabulário
from collections import Counter
import numpy as np

# Top palavras mais frequentes
all_tokens = []
for tokens in df_clean['tokens']:
    all_tokens.extend(tokens)

word_freq = Counter(all_tokens)
top_words = word_freq.most_common(30)

print("\n📝 Top 30 palavras mais frequentes:")
for word, count in top_words:
    print(f"  {word:20s}: {count:6,}")

print(f"\n📊 Vocabulário único: {len(word_freq):,} palavras")
print(f"📊 Total de tokens:   {len(all_tokens):,}")

# Visualizar top 20
words, counts = zip(*top_words[:20])
plt.figure(figsize=(12, 6))
plt.barh(range(len(words)), counts, color='steelblue')
plt.yticks(range(len(words)), words)
plt.xlabel('Frequência')
plt.title('Top 20 Palavras Mais Frequentes')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# 3. Análise de sentiment
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 3.1 Distribuição de sentiment
df_clean['sentiment'].hist(bins=50, ax=axes[0, 0], color='skyblue', edgecolor='black')
axes[0, 0].set_title('Distribuição de Sentiment')
axes[0, 0].set_xlabel('Sentiment Score')
axes[0, 0].set_ylabel('Frequência')
axes[0, 0].axvline(0, color='red', linestyle='--', alpha=0.7)

# 3.2 Credibility score
df_clean['credibility_score'].hist(bins=30, ax=axes[0, 1], color='lightgreen', edgecolor='black')
axes[0, 1].set_title('Distribuição de Credibility Score')
axes[0, 1].set_xlabel('Credibility')
axes[0, 1].set_ylabel('Frequência')

# 3.3 Novelty score temporal
df_clean['published_at_date'] = pd.to_datetime(df_clean['published_at'])
df_novelty_time = df_clean.groupby(df_clean['published_at_date'].dt.to_period('M'))['novelty_score'].mean()
df_novelty_time.plot(ax=axes[1, 0], color='coral', marker='o')
axes[1, 0].set_title('Novelty Score ao Longo do Tempo')
axes[1, 0].set_xlabel('Mês')
axes[1, 0].set_ylabel('Novelty Médio')
axes[1, 0].grid(True, alpha=0.3)

# 3.4 Sentiment por fonte
df_sent_source = df_clean.groupby('source_type')['sentiment'].mean().sort_values()
df_sent_source.plot(kind='barh', ax=axes[1, 1], color='mediumpurple')
axes[1, 1].set_title('Sentiment Médio por Tipo de Fonte')
axes[1, 1].set_xlabel('Sentiment Médio')
axes[1, 1].axvline(0, color='red', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

print("\n📊 Estatísticas dos features:")
print(df_clean[['sentiment', 'credibility_score', 'novelty_score', 'token_count']].describe())

## 📊 Análise dos Features Gerados

In [ ]:
# 2. Executar pipeline de preprocessamento
from src.pipeline.preprocess_pipeline import run_preprocess_pipeline

# Configuração:
# - generate_emb=True: Gera embeddings (LENTO ~10-30min para 30k notícias)
# - analyze_sent=True: Análise de sentimento
#
# Para execução rápida de teste, use generate_emb=False

df_clean = run_preprocess_pipeline(
    input_file=None,  # Busca automaticamente news_multisource.parquet
    generate_emb=True,  # ATENÇÃO: Pode demorar muito!
    analyze_sent=True
)

print(f"\n✅ Preprocessamento concluído: {len(df_clean):,} registros")

In [ ]:
# 1. Setup
import os
import sys
from pathlib import Path

project_root = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(project_root))

print(f"📂 Project root: {project_root}")

# 14. Preprocessamento Avançado PT-BR

**Objetivo**: Processar textos em português com:

1. **Limpeza** (HTML, URLs, stopwords, normalização)
2. **Embeddings** (SentenceTransformer 768-dim)
3. **Sentiment** (VADER + keywords financeiras)
4. **Credibility Score** (baseado em fonte e características)
5. **Novelty Score** (decay temporal)
6. **BoW Diário** (agregação por data)

**Outputs**: 
- `news_clean.parquet` (dataset completo)
- `bow_daily.parquet` (agregação diária)